# Install TabPFN Extenstions

In [ ]:
import importlib.util
import sys

def _missing(mod):
    return importlib.util.find_spec(mod) is None

to_install = []
# `shap` is only needed for the plotting API in the SHAP section — shapiq
# (installed via tabpfn-extensions[all]) does the actual computation.
if _missing("shap"):
    to_install.append("shap")

if to_install:
    print("Installing:", to_install)
    get_ipython().run_line_magic("pip", "install -q --no-warn-conflicts " + " ".join(to_install))
    print("Done. If imports fail, restart the kernel and re-run.")
else:
    print("shap already available.")

In [ ]:
print("Installing tabpfn-extensions...")
# Clone and install the repository
!pip install "tabpfn-extensions[all] @ git+https://github.com/PriorLabs/tabpfn-extensions.git"
print("Done. If imports fail, restart the kernel and re-run.")

### Load env

In [ ]:
import os

# Auto-detect Kaggle. Uncomment the next line if you copied this notebook and detection fails.
is_kaggle_env = os.path.isdir("/kaggle/working")
# is_kaggle_env = True

try:
    import torch

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
except Exception:
    DEVICE = "cpu"
    DEVICE_NAME = "CPU"

print(f"Runtime: {'Kaggle' if is_kaggle_env else 'local'} | TabPFN device: {DEVICE} ({DEVICE_NAME})")

### Load data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_MODE = "raw"            # "raw" (spec default) or "processed"

TARGET_COL = "Stent thrombosis"
DROP_FEATURES = ["Time since stent implantation"]   # leakage control
ID_COLS = ["NO.", "Name"]
RANDOM_STATE = 42
TEST_SIZE = 0.3

# --- TabPFN inference knobs ---
N_ESTIMATORS = -1                # ensemble of forward passes (not trees); more = steadier probs
BALANCE_PROBABILITIES = True     # helps under the strong VLST class imbalance
IGNORE_PRETRAINING_LIMITS = False # allow n_train / n_features outside default range

KAGGLE_RAW_CSV = "/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv"
KAGGLE_PROCESSED_DIR = "/kaggle/input/datasets/amirmahdidaraei/preprocessed-data"
KAGGLE_RESULT_SUBDIR = "modeling_tabpfn"
RESULT_DIR = None  # resolved in the load cell

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "data" / "raw" / "VLST.csv").is_file():
            return d
    raise FileNotFoundError(
        "Could not locate data/raw/VLST.csv above the current working directory."
    )


def _discover_vlst_csv() -> Path:
    """Resolve VLST.csv on Kaggle (env override, then recursive search under /kaggle/input)."""
    env = os.environ.get("VLST_RAW_CSV")
    if env and Path(env).is_file():
        return Path(env)

    candidates = [
        Path(KAGGLE_RAW_CSV),
        Path("/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv"),
        Path("/kaggle/input/vlst-data/VLST.csv"),
        Path("/kaggle/input/VLST_data/VLST.csv"),
    ]
    for p in candidates:
        if p.is_file():
            return p

    base = Path("/kaggle/input")
    if base.is_dir():
        for p in base.rglob("VLST.csv"):
            if p.is_file():
                return p

    raise FileNotFoundError(
        "VLST.csv not found on Kaggle. Upload it as a dataset (see §0) or set "
        "os.environ['VLST_RAW_CSV'] = '/kaggle/input/<dataset>/VLST.csv'."
    )


def _resolve_paths():
    """Return (raw_path, processed_dir, result_dir, label) for local or Kaggle."""
    if is_kaggle_env:
        raw = _discover_vlst_csv()
        processed = Path(
            os.environ.get("VLST_PROCESSED_DIR", KAGGLE_PROCESSED_DIR)
        )
        result = Path(
            os.environ.get(
                "VLST_RESULT_DIR",
                str(Path("/kaggle/working") / KAGGLE_RESULT_SUBDIR),
            )
        )
        return raw, processed, result, f"Kaggle | raw={raw}"

    repo = _find_repo_root()
    return (
        repo / "data" / "raw" / "VLST.csv",
        repo / "data" / "processed",
        repo / "data" / "result" / "modeling_tabpfn",
        f"local | repo={repo}",
    )


RAW_PATH, PROCESSED_DIR, RESULT_DIR, _path_label = _resolve_paths()
RESULT_DIR = Path(RESULT_DIR)
PROCESSED_DIR = Path(PROCESSED_DIR)
RAW_PATH = Path(RAW_PATH)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(_path_label)
print("RAW_PATH:", RAW_PATH)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("RESULT_DIR:", RESULT_DIR)


def load_raw():
    """Minimal, TabPFN-native handling: keep NaNs, code text columns, no scaling/one-hot."""
    df = pd.read_csv(RAW_PATH)
    df = df.drop(columns=[c for c in ID_COLS if c in df.columns])
    y = df[TARGET_COL].astype(int).to_numpy()
    drop = [TARGET_COL] + [c for c in DROP_FEATURES if c in df.columns]
    X_df = df.drop(columns=drop)
    for c in X_df.columns:
        if X_df[c].dtype == object:
            coerced = pd.to_numeric(X_df[c].astype(str).str.strip(), errors="coerce")
            if coerced.notna().mean() >= 0.5:        # genuinely numeric (e.g. "21.00 ")
                X_df[c] = coerced
            else:                                     # categorical text -> integer codes (NOT one-hot)
                codes = X_df[c].astype("category").cat.codes.astype(float)
                X_df[c] = codes.where(codes >= 0, np.nan)
    return X_df.to_numpy(dtype=float), y, list(X_df.columns)

X_all, y_all, feature_names = load_raw()
print("Loaded RAW VLST.csv")

print(f"X: {X_all.shape} | y: {y_all.shape} | Features: {len(feature_names)}")

### Feature Selection Part

In [ ]:
"""Feature selection on the VLST dataset with TabPFN.

WARNING: This step may run slowly on CPU-only systems. Prefer running with
GPU acceleration — feature selection runs many TabPFN fits per round.

We keep `n_estimators=1` (a single forward pass per fit) so the total
wallclock stays manageable; the goal is a stable *ranking*, not the
tightest absolute probabilities. Bump `n_estimators` when you have GPU
budget to spare.
"""

from tabpfn_extensions import TabPFNClassifier, interpretability

# Feature selection is CV-driven internally, so we pass the full pool
# (X_all, y_all, feature_names) loaded above — no train/test split needed.
clf = TabPFNClassifier(
    n_estimators=1,
    balance_probabilities=BALANCE_PROBABILITIES,
    ignore_pretraining_limits=IGNORE_PRETRAINING_LIMITS,
    random_state=RANDOM_STATE,
)

N_FEATURES_TO_SELECT = min(20, X_all.shape[1])

# With verbose=True (the default) the wrapper prints the baseline CV score
# on all features, the per-round picks, and the selected names + CV score
# on the subset. The same numbers are also available on the returned
# FeatureSelectionResult for programmatic use.
result = interpretability.feature_selection.feature_selection(
    estimator=clf,
    X=X_all,
    y=y_all,
    n_features_to_select=N_FEATURES_TO_SELECT,
    feature_names=list(feature_names),
)

# `result.selected_names` is populated because we passed `feature_names`.
# `result.selector.transform(X_all)` would project to just those columns;
# `result.support_mask` / `result.selected_indices` are also available.
print("\nProgrammatic summary:")
print(f"Selected features ({len(result.selected_names)}): {result.selected_names}")
print(
    f"CV score before / after: "
    f"{result.baseline_score_mean:.4f} -> {result.selected_score_mean:.4f}"
)

# Persist the selected feature list next to the other TabPFN artifacts.
fs_out = RESULT_DIR / "interpretability_feature_selection.csv"
pd.DataFrame(
    {
        "rank": range(1, len(result.selected_names) + 1),
        "feature": result.selected_names,
    },
).to_csv(fs_out, index=False)
print(f"Saved: {fs_out}")

### PDP (Partial Dependence Plot) Part

In [ ]:
"""Partial dependence plots for the VLST TabPFN classifier.

PDP issues many predicts (`grid_resolution * n_features` for 1D plots;
more for 2D interactions) against the same fitted model, so the KV cache
(`fit_mode='fit_with_cache'`) avoids redoing the encoder pass over
X_train on every grid point. Falls back to the default constructor for
backends/versions that don't support the cache — that's tabpfn-client
(TypeError on the kwarg) and older local tabpfn (accepts the kwarg but
raises ValueError/NotImplementedError at fit time).
"""

import warnings

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from tabpfn_extensions import TabPFNClassifier
from tabpfn_extensions.interpretability.pdp import partial_dependence_plots

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE,
    stratify=y_all,           # VLST is imbalanced — always stratify
    random_state=RANDOM_STATE,
)

def _make_pdp_clf(**extra):
    return TabPFNClassifier(
        n_estimators=N_ESTIMATORS,
        balance_probabilities=BALANCE_PROBABILITIES,
        ignore_pretraining_limits=IGNORE_PRETRAINING_LIMITS,
        random_state=RANDOM_STATE,
        **extra,
    )

try:
    clf = _make_pdp_clf(fit_mode="fit_with_cache")
    clf.fit(X_train, y_train)
    if hasattr(clf, "executor_"):
        clf.executor_.keep_cache_on_device = True
except (TypeError, ValueError, NotImplementedError):
    warnings.warn(
        "PDP would benefit substantially from the KV cache, but the "
        "current TabPFN install doesn't support fit_mode='fit_with_cache' "
        "(typical of older tabpfn versions or the tabpfn-client backend). "
        "Upgrade to the latest version of tabpfn (`pip install -U tabpfn`) "
        "for a substantial speedup on this example. Falling back to the "
        "default constructor.",
        UserWarning,
        stacklevel=2,
    )
    clf = _make_pdp_clf()
    clf.fit(X_train, y_train)

# 1D PD for the first 3 features + one pairwise interaction as a smoke
# test. Swap these for the winners of the Feature Selection section above
# once you have that ranking.
pd_features = [0, 1, 2, (0, 3)]

disp = partial_dependence_plots(
    estimator=clf,
    X=X_test,
    features=pd_features,
    grid_resolution=30,
    kind="average",
    target_class=1,       # positive class = "Stent thrombosis"
    feature_names=list(feature_names),
)
disp.figure_.suptitle("Partial dependence — VLST (P[Stent thrombosis])")

pdp_out = RESULT_DIR / "interpretability_pdp.png"
plt.savefig(pdp_out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {pdp_out}")

### SHAP Part

In [ ]:
"""SHAP values for the VLST TabPFN classifier via shapiq, visualized with
the SHAP library's plotting API.

We use shapiq for the actual Shapley-value computation (faster and
extension-friendly for TabPFN) but the SHAP library's plotting ecosystem
is mature. This bridges the two: wrap shapiq output in a
`shap.Explanation` and call `shap.plots.*` / `shap.summary_plot`.

For a classifier, `get_tabpfn_imputation_explainer` defaults to
`class_index=1` — the positive class. That's what we want on VLST (target
= "Stent thrombosis").

The `shap` package is not part of the `interpretability` extra (we depend
on shapiq for compute). It's installed by the top-of-notebook install
cell.

VLST has ~50 features, so 2**d exact enumeration is astronomical — the
budget below samples coalitions instead. Increase it on GPU for tighter
estimates.

The TabPFN model is constructed with `fit_mode="fit_with_cache"` to
engage the KV cache, which speeds up Shapley-value computation by one to
two orders of magnitude; the shapiq wrapper warns if the cache isn't
enabled.
"""

from __future__ import annotations

import warnings

import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split

from tabpfn_extensions import TabPFNClassifier
from tabpfn_extensions.interpretability import (
    shapiq as tabpfn_shapiq,
    shapiq_to_shap_explanation,
)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=RANDOM_STATE,
)

# Explain positives first so the beeswarm/waterfall highlight the
# minority (informative) class before falling back to negatives.
_pos_idx = np.where(y_test == 1)[0]
_neg_idx = np.where(y_test == 0)[0]
SHAP_N_EXPLAIN = min(30, X_test.shape[0])
_order = np.concatenate([_pos_idx, _neg_idx])[:SHAP_N_EXPLAIN]
X_explain = X_test[_order]

def _make_shap_clf(**extra):
    return TabPFNClassifier(
        n_estimators=N_ESTIMATORS,
        balance_probabilities=BALANCE_PROBABILITIES,
        ignore_pretraining_limits=IGNORE_PRETRAINING_LIMITS,
        random_state=RANDOM_STATE,
        **extra,
    )

try:
    clf = _make_shap_clf(fit_mode="fit_with_cache")
    clf.fit(X_train, y_train)
except (TypeError, ValueError, NotImplementedError):
    warnings.warn(
        "SHAP would benefit substantially from the KV cache, but the "
        "current TabPFN install doesn't support fit_mode='fit_with_cache' "
        "(typical of older tabpfn versions or the tabpfn-client backend). "
        "Falling back to the default constructor.",
        UserWarning, stacklevel=2,
    )
    clf = _make_shap_clf()
    clf.fit(X_train, y_train)

# `class_index` defaults to 1 for classifiers — explains P(positive).
explainer = tabpfn_shapiq.get_tabpfn_imputation_explainer(
    model=clf,
    data=X_train,
    index="SV",
    max_order=1,
)

SHAPIQ_BUDGET = 1024

# `shapiq_to_shap_explanation` runs one .explain() call per row, stacks the
# (d,) arrays, averages baseline values, and packages everything for the
# SHAP plotting API.
print(f"Computing Shapley values for {SHAP_N_EXPLAIN} rows (budget={SHAPIQ_BUDGET})...")
explanation = shapiq_to_shap_explanation(
    explainer,
    X_explain,
    budget=SHAPIQ_BUDGET,
    feature_names=list(feature_names),
)

# 1. Summary plot — beeswarm of feature attributions across all explained rows.
shap.summary_plot(explanation, show=False)
plt.savefig(RESULT_DIR / "interpretability_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

# 2. Scatter — SHAP value of feature 0 vs. its raw value, colored by the
#    feature shap picks as its strongest interaction partner.
shap.plots.scatter(explanation[:, 0], show=False)
plt.savefig(RESULT_DIR / "interpretability_shap_scatter_f0.png", dpi=150, bbox_inches="tight")
plt.show()

# 3. Bar plot — mean(|SHAP|) ranking of features.
shap.plots.bar(explanation, show=False)
plt.savefig(RESULT_DIR / "interpretability_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

# 4. Beeswarm plot — same data as summary, new-API styling.
shap.plots.beeswarm(explanation, show=False)
plt.savefig(RESULT_DIR / "interpretability_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

# 5. Waterfall plot — explain a single row (E[f(X)] -> f(x) breakdown).
shap.plots.waterfall(explanation[0], show=False)
plt.savefig(RESULT_DIR / "interpretability_shap_waterfall_row0.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"SHAP plots saved under: {RESULT_DIR}")

### SHAP-IQ (Shapley Interaction Quantification) Part

In [ ]:
"""Shapley values + pairwise Shapley interactions for the VLST TabPFN
classifier using the shapiq library, visualized with shapiq's native
plots.

Two paradigms for "feature removal" are illustrated:

  1. Imputation-based (`get_tabpfn_imputation_explainer`): masked
     features are filled by an imputer (default: baseline — mean for
     numeric features, mode for categorical, learned from `data`). The
     training set is fixed across coalitions, so the KV-cache fast path
     applies — construct the model with `fit_mode="fit_with_cache"`.

  2. Remove-and-recontextualize (`get_tabpfn_explainer`): TabPFN is
     re-fit on each coalition's column subset. Does not benefit from
     the KV cache (one predict per fit). Left commented out below —
     prohibitive at VLST's dimensionality.

For classifiers, both explainers default to `class_index=1`, i.e. they
explain P(positive) — "Stent thrombosis" on VLST.
"""

from __future__ import annotations

import warnings

from sklearn.model_selection import train_test_split

from tabpfn_extensions import TabPFNClassifier
from tabpfn_extensions.interpretability import shapiq as tabpfn_shapiq

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=RANDOM_STATE,
)

# Interaction/network/upset plots are most informative on a minority-class
# row when one exists; fall back to the first row otherwise.
_pos = np.where(y_test == 1)[0]
x_explain = X_test[_pos[0] if len(_pos) else 0]

def _make_iq_clf(**extra):
    return TabPFNClassifier(
        n_estimators=N_ESTIMATORS,
        balance_probabilities=BALANCE_PROBABILITIES,
        ignore_pretraining_limits=IGNORE_PRETRAINING_LIMITS,
        random_state=RANDOM_STATE,
        **extra,
    )

try:
    clf = _make_iq_clf(fit_mode="fit_with_cache")
    clf.fit(X_train, y_train)
except (TypeError, ValueError, NotImplementedError):
    warnings.warn(
        "shapiq would benefit substantially from the KV cache, but the "
        "current TabPFN install doesn't support fit_mode='fit_with_cache'. "
        "Falling back to the default constructor.",
        UserWarning, stacklevel=2,
    )
    clf = _make_iq_clf()
    clf.fit(X_train, y_train)

# VLST has ~50 features; 2**d exact enumeration is intractable — use
# sampled coalitions. Interactions (max_order=2) generally need a bigger
# budget than plain SV.
SV_BUDGET = 1024
KSII_BUDGET = 2048


# -----------------------------------------------------------------------------
# 1. Imputation-based explainer (uses the KV cache)
# -----------------------------------------------------------------------------
imputation_explainer = tabpfn_shapiq.get_tabpfn_imputation_explainer(
    model=clf,
    data=X_train,
    index="SV",     # plain Shapley values
    max_order=1,
)
print(f"Computing imputation-based Shapley values (budget={SV_BUDGET})...")
sv_imp = imputation_explainer.explain(x=x_explain, budget=SV_BUDGET)
sv_imp.plot_force(feature_names=list(feature_names))


# -----------------------------------------------------------------------------
# 2. Pairwise Shapley interactions via the same explainer (k-SII, max_order=2)
# -----------------------------------------------------------------------------
interaction_explainer = tabpfn_shapiq.get_tabpfn_imputation_explainer(
    model=clf,
    data=X_train,
    index="k-SII",  # k-Shapley Interaction Index — extends SHAP to interactions
    max_order=2,
)
print(f"Computing pairwise Shapley interactions (k-SII, budget={KSII_BUDGET})...")
iv_interactions = interaction_explainer.explain(x=x_explain, budget=KSII_BUDGET)

# Network plot: features as nodes sized by individual SV; edges colored by
# pairwise interaction strength. Specific to shapiq (not in the shap library).
iv_interactions.plot_network(feature_names=list(feature_names))

# Upset plot of top interactions.
iv_interactions.plot_upset(feature_names=list(feature_names))


# -----------------------------------------------------------------------------
# 3. Remove-and-recontextualize (Rundel) — much slower, no KV cache.
# -----------------------------------------------------------------------------
# Each coalition triggers a fresh TabPFN fit on a different column subset,
# followed by exactly one predict. On VLST-sized d this is prohibitive.
# Kept as reference — uncomment only for small feature subsets.
#
#     rundel_explainer = tabpfn_shapiq.get_tabpfn_explainer(
#         model=clf,
#         data=X_train,
#         labels=y_train,
#         index="SV",
#         max_order=1,
#     )
#     sv_rundel = rundel_explainer.explain(x=x_explain, budget=SV_BUDGET)
#     sv_rundel.plot_force(feature_names=list(feature_names))